# 第23章：FlashAttention v4 — 最新进展

## 前置知识
- 第19-22章

## 学习目标
- 理解 GQA（Grouped Query Attention）并实现
- 学习变长序列（Variable-Length）的处理方式
- 掌握滑动窗口注意力（Sliding Window Attention）
- 了解块稀疏注意力和 PagedAttention 的概念
- 展望注意力优化的未来方向

In [1]:
import torch
import triton
import triton.language as tl
import math

## 23.1 GQA — Grouped Query Attention

现代 LLM（LLaMA 2/3, Mistral, Gemma）广泛使用 GQA：

```
MHA (Multi-Head Attention):     GQA (Grouped Query Attention):
Q heads: H₀ H₁ H₂ H₃ H₄ H₅   Q heads: H₀ H₁ H₂ H₃ H₄ H₅
K heads: K₀ K₁ K₂ K₃ K₄ K₅   K heads: K₀    K₁    K₂
V heads: V₀ V₁ V₂ V₃ V₄ V₅   V heads: V₀    V₁    V₂
         ↕  ↕  ↕  ↕  ↕  ↕              ↕ ↕   ↕ ↕   ↕ ↕
         1:1 对应                       2个Q头共享1个KV头

MQA (Multi-Query Attention):    GQA 介于 MHA 和 MQA 之间:
Q heads: H₀ H₁ H₂ H₃ H₄ H₅   - 减少 KV cache 内存
K heads: K₀                     - 保持大部分精度
V heads: V₀                     - LLaMA 2 70B: 8个KV头, 64个Q头
         ↕ ↕ ↕ ↕ ↕ ↕
         所有Q头共享1个KV头
```

### GQA 的关键变化

```python
# MHA: num_q_heads == num_kv_heads
# GQA: num_q_heads = num_kv_heads * group_size
# MQA: num_kv_heads = 1

# Q 头索引 → KV 头索引的映射:
kv_head_idx = q_head_idx // group_size
```

In [2]:
@triton.jit
def flash_attention_gqa_kernel(
    Q, K, V, Out,
    sm_scale,
    stride_qb, stride_qh, stride_qm, stride_qk,
    stride_kb, stride_kh, stride_kn, stride_kk,
    stride_vb, stride_vh, stride_vn, stride_vk,
    stride_ob, stride_oh, stride_om, stride_ok,
    B, H_Q, H_KV, N, D,
    GQA_GROUP_SIZE: tl.constexpr,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_D: tl.constexpr,
    IS_CAUSAL: tl.constexpr,
):
    """
    FlashAttention with GQA support.
    
    Q: (B, H_Q, N, D)
    K: (B, H_KV, N, D)    H_Q = H_KV * GQA_GROUP_SIZE
    V: (B, H_KV, N, D)
    """
    # Program ID: 处理哪个 Q 块
    pid_m = tl.program_id(0)     # Q 序列块索引
    pid_bh = tl.program_id(1)    # batch * Q_head 联合索引
    
    # 分解 batch 和 head 索引
    batch_idx = pid_bh // H_Q
    q_head_idx = pid_bh % H_Q
    # GQA: Q head → KV head 映射
    kv_head_idx = q_head_idx // GQA_GROUP_SIZE
    
    # 偏移量
    q_offset = batch_idx * stride_qb + q_head_idx * stride_qh
    k_offset = batch_idx * stride_kb + kv_head_idx * stride_kh  # 注意: 用 kv_head_idx
    v_offset = batch_idx * stride_vb + kv_head_idx * stride_vh
    o_offset = batch_idx * stride_ob + q_head_idx * stride_oh
    
    # Q 块的行偏移
    m_offsets = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    d_offsets = tl.arange(0, BLOCK_D)
    
    # 加载 Q 块 (BLOCK_M x D)
    q_ptrs = Q + q_offset + m_offsets[:, None] * stride_qm + d_offsets[None, :] * stride_qk
    q_mask = (m_offsets[:, None] < N) & (d_offsets[None, :] < D)
    q = tl.load(q_ptrs, mask=q_mask, other=0.0)
    
    # 初始化在线 softmax 状态
    m_i = tl.full((BLOCK_M,), value=float('-inf'), dtype=tl.float32)
    l_i = tl.zeros((BLOCK_M,), dtype=tl.float32)
    acc = tl.zeros((BLOCK_M, BLOCK_D), dtype=tl.float32)
    
    # Causal: 只需要迭代到 q_position + 1
    if IS_CAUSAL:
        kv_end = min((pid_m + 1) * BLOCK_M, N)
    else:
        kv_end = N
    
    # 内循环: 遍历 K,V 块
    for n_start in range(0, kv_end, BLOCK_N):
        n_offsets = n_start + tl.arange(0, BLOCK_N)
        
        # 加载 K 块 (BLOCK_N x D)
        k_ptrs = K + k_offset + n_offsets[:, None] * stride_kn + d_offsets[None, :] * stride_kk
        k_mask = (n_offsets[:, None] < N) & (d_offsets[None, :] < D)
        k = tl.load(k_ptrs, mask=k_mask, other=0.0)
        
        # S = Q @ K^T * scale  (BLOCK_M x BLOCK_N)
        s = tl.dot(q, tl.trans(k)) * sm_scale
        
        # Causal mask
        if IS_CAUSAL:
            causal_mask = m_offsets[:, None] >= n_offsets[None, :]
            s = tl.where(causal_mask, s, float('-inf'))
        
        # 边界 mask
        s = tl.where(n_offsets[None, :] < N, s, float('-inf'))
        
        # Online softmax 更新
        m_ij = tl.max(s, axis=1)  # (BLOCK_M,)
        m_new = tl.maximum(m_i, m_ij)
        alpha = tl.exp(m_i - m_new)
        p = tl.exp(s - m_new[:, None])
        
        l_i = l_i * alpha + tl.sum(p, axis=1)
        acc = acc * alpha[:, None]
        
        # 加载 V 块并累加
        v_ptrs = V + v_offset + n_offsets[:, None] * stride_vn + d_offsets[None, :] * stride_vk
        v_mask = (n_offsets[:, None] < N) & (d_offsets[None, :] < D)
        v = tl.load(v_ptrs, mask=v_mask, other=0.0)
        acc += tl.dot(p.to(v.dtype), v)
        
        m_i = m_new
    
    # 最终归一化
    acc = acc / l_i[:, None]
    
    # 存储输出
    o_ptrs = Out + o_offset + m_offsets[:, None] * stride_om + d_offsets[None, :] * stride_ok
    o_mask = (m_offsets[:, None] < N) & (d_offsets[None, :] < D)
    tl.store(o_ptrs, acc.to(Out.dtype.element_ty), mask=o_mask)

In [3]:
def flash_attention_gqa(q, k, v, causal=False):
    """
    FlashAttention with GQA.
    q: (B, H_Q, N, D)
    k: (B, H_KV, N, D)
    v: (B, H_KV, N, D)
    """
    B, H_Q, N, D = q.shape
    _, H_KV, _, _ = k.shape
    assert H_Q % H_KV == 0, "H_Q must be divisible by H_KV"
    GQA_GROUP_SIZE = H_Q // H_KV
    
    sm_scale = 1.0 / math.sqrt(D)
    out = torch.empty_like(q)
    
    BLOCK_M = 64
    BLOCK_N = 64
    BLOCK_D = triton.next_power_of_2(D)
    
    grid = (triton.cdiv(N, BLOCK_M), B * H_Q)
    
    flash_attention_gqa_kernel[grid](
        q, k, v, out,
        sm_scale,
        q.stride(0), q.stride(1), q.stride(2), q.stride(3),
        k.stride(0), k.stride(1), k.stride(2), k.stride(3),
        v.stride(0), v.stride(1), v.stride(2), v.stride(3),
        out.stride(0), out.stride(1), out.stride(2), out.stride(3),
        B, H_Q, H_KV, N, D,
        GQA_GROUP_SIZE=GQA_GROUP_SIZE,
        BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N, BLOCK_D=BLOCK_D,
        IS_CAUSAL=causal,
    )
    return out

# 测试 GQA
B, H_Q, H_KV, N, D = 2, 8, 2, 128, 64  # 4个Q头共享1个KV头
q = torch.randn(B, H_Q, N, D, device='cuda', dtype=torch.float16)
k = torch.randn(B, H_KV, N, D, device='cuda', dtype=torch.float16)
v = torch.randn(B, H_KV, N, D, device='cuda', dtype=torch.float16)

out = flash_attention_gqa(q, k, v, causal=True)

# 参考实现
GQA_GROUP = H_Q // H_KV
k_expanded = k.repeat_interleave(GQA_GROUP, dim=1)  # (B, H_Q, N, D)
v_expanded = v.repeat_interleave(GQA_GROUP, dim=1)
s = (q.float() @ k_expanded.float().transpose(-2, -1)) / math.sqrt(D)
causal_mask = torch.triu(torch.ones(N, N, device='cuda'), diagonal=1).bool()
s.masked_fill_(causal_mask, float('-inf'))
p = torch.softmax(s, dim=-1)
ref = (p @ v_expanded.float()).half()

print(f"GQA FlashAttention:")
print(f"  Q heads: {H_Q}, KV heads: {H_KV}, Group size: {GQA_GROUP}")
print(f"  形状: Q({B},{H_Q},{N},{D}), K({B},{H_KV},{N},{D})")
print(f"  正确性: {torch.allclose(out, ref, atol=1e-1)}")
print(f"  最大误差: {(out.float() - ref.float()).abs().max().item():.4f}")

GQA FlashAttention:
  Q heads: 8, KV heads: 2, Group size: 4
  形状: Q(2,8,128,64), K(2,2,128,64)
  正确性: True
  最大误差: 0.0010


## 23.2 变长序列 (Variable-Length Sequences)

在实际推理中，一个 batch 内的序列长度通常不同：

```
Padding 方案（浪费计算）:
  seq 0: [tok tok tok tok PAD PAD PAD PAD]  长度=4, padding=4
  seq 1: [tok tok tok tok tok tok PAD PAD]  长度=6, padding=2
  seq 2: [tok tok PAD PAD PAD PAD PAD PAD]  长度=2, padding=6
                                            浪费: 12/24 = 50%

Varlen 方案（紧凑存储）:
  连续存储: [seq0_tok×4 | seq1_tok×6 | seq2_tok×2]
  cu_seqlens: [0, 4, 10, 12]  ← 累计序列长度
                                 无浪费!
```

### cu_seqlens 的含义
```python
cu_seqlens = [0, 4, 10, 12]
# 序列 0: tokens[0:4]    长度 = cu_seqlens[1] - cu_seqlens[0] = 4
# 序列 1: tokens[4:10]   长度 = cu_seqlens[2] - cu_seqlens[1] = 6
# 序列 2: tokens[10:12]  长度 = cu_seqlens[3] - cu_seqlens[2] = 2
```

In [4]:
def flash_attention_varlen_reference(q, k, v, cu_seqlens, max_seqlen):
    """
    变长序列 Attention 参考实现。
    
    q, k, v: (total_tokens, H, D) — 所有序列紧凑拼接
    cu_seqlens: (B+1,) — 累计序列长度
    """
    B = cu_seqlens.shape[0] - 1
    H = q.shape[1]
    D = q.shape[2]
    sm_scale = 1.0 / math.sqrt(D)
    
    output = torch.zeros_like(q)
    
    for i in range(B):
        start = cu_seqlens[i].item()
        end = cu_seqlens[i + 1].item()
        seq_len = end - start
        
        qi = q[start:end]  # (seq_len, H, D)
        ki = k[start:end]
        vi = v[start:end]
        
        # 标准 attention
        for h in range(H):
            s = qi[:, h, :].float() @ ki[:, h, :].float().T * sm_scale
            # Causal mask
            mask = torch.triu(torch.ones(seq_len, seq_len, device=q.device), diagonal=1).bool()
            s.masked_fill_(mask, float('-inf'))
            p = torch.softmax(s, dim=-1)
            output[start:end, h, :] = (p @ vi[:, h, :].float()).to(q.dtype)
    
    return output

# 测试变长序列
H, D = 4, 64
seq_lengths = [32, 64, 16]  # 三个不同长度的序列
total_tokens = sum(seq_lengths)
cu_seqlens = torch.tensor([0] + list(torch.cumsum(torch.tensor(seq_lengths), 0)), 
                           device='cuda', dtype=torch.int32)

q = torch.randn(total_tokens, H, D, device='cuda', dtype=torch.float16)
k = torch.randn(total_tokens, H, D, device='cuda', dtype=torch.float16)
v = torch.randn(total_tokens, H, D, device='cuda', dtype=torch.float16)

out = flash_attention_varlen_reference(q, k, v, cu_seqlens, max(seq_lengths))

print(f"变长序列 Attention:")
print(f"  序列长度: {seq_lengths}")
print(f"  总 token 数: {total_tokens}")
print(f"  cu_seqlens: {cu_seqlens.tolist()}")
print(f"  输出形状: {out.shape}")
print(f"  (实际 Triton 优化版需要在 kernel 中处理 cu_seqlens)")

变长序列 Attention:
  序列长度: [32, 64, 16]
  总 token 数: 112
  cu_seqlens: [0, 32, 96, 112]
  输出形状: torch.Size([112, 4, 64])
  (实际 Triton 优化版需要在 kernel 中处理 cu_seqlens)


## 23.3 滑动窗口注意力 (Sliding Window Attention)

Mistral 等模型使用滑动窗口限制注意力范围：

```
全注意力 (N=8):           滑动窗口 (window=3):
Q\K  0 1 2 3 4 5 6 7     Q\K  0 1 2 3 4 5 6 7
 0 [ ✓ . . . . . . . ]    0 [ ✓ . . . . . . . ]
 1 [ ✓ ✓ . . . . . . ]    1 [ ✓ ✓ . . . . . . ]
 2 [ ✓ ✓ ✓ . . . . . ]    2 [ ✓ ✓ ✓ . . . . . ]
 3 [ ✓ ✓ ✓ ✓ . . . . ]    3 [ . ✓ ✓ ✓ . . . . ]
 4 [ ✓ ✓ ✓ ✓ ✓ . . . ]    4 [ . . ✓ ✓ ✓ . . . ]
 5 [ ✓ ✓ ✓ ✓ ✓ ✓ . . ]    5 [ . . . ✓ ✓ ✓ . . ]
 6 [ ✓ ✓ ✓ ✓ ✓ ✓ ✓ . ]    6 [ . . . . ✓ ✓ ✓ . ]
 7 [ ✓ ✓ ✓ ✓ ✓ ✓ ✓ ✓ ]    7 [ . . . . . ✓ ✓ ✓ ]

    O(N²) 复杂度              O(N×W) 复杂度, W=窗口大小
```

在 FlashAttention 中实现滑动窗口只需要修改 K 块的迭代范围：

```python
# 标准 causal:
for n_start in range(0, min((pid_m + 1) * BLOCK_M, N), BLOCK_N):

# 滑动窗口 causal:
kv_start = max(0, pid_m * BLOCK_M - window_size)
kv_end = min((pid_m + 1) * BLOCK_M, N)
for n_start in range(kv_start, kv_end, BLOCK_N):
```

## 23.4 块稀疏注意力 (Block-Sparse Attention)

对于超长序列，可以用结构化稀疏模式跳过不重要的注意力块：

```
Block Mask (每个块 64×64):
┌──┬──┬──┬──┬──┬──┬──┬──┐
│██│  │  │  │  │  │  │  │  block_mask[i][j] = 1 → 计算
│██│██│  │  │  │  │  │  │  block_mask[i][j] = 0 → 跳过
│██│██│██│  │  │  │  │  │
│  │██│██│██│  │  │  │  │  稀疏模式:
│  │  │██│██│██│  │  │  │  - 因果 (下三角)
│  │  │  │██│██│██│  │  │  - 局部窗口
│  │  │  │  │██│██│██│  │  - 全局 token
│  │  │  │  │  │██│██│██│  - 自定义模式
└──┴──┴──┴──┴──┴──┴──┴──┘

在 FlashAttention 循环中:
for n_block in range(num_kv_blocks):
    if block_mask[m_block, n_block]:  # 只计算非零块
        compute_attention_block(...)
```

## 23.5 PagedAttention — 推理时的 KV Cache 管理

在 LLM 推理中，KV Cache 是主要的内存瓶颈：

```
传统 KV Cache:              PagedAttention:
┌──────────────────┐        ┌──────────────────┐
│ Seq 0: 连续分配   │        │ Page Table:       │
│ [████████████    ]│  →     │ Seq 0: [P2,P5,P1]│
│     浪费空间 ↑    │        │ Seq 1: [P3,P0]   │
│ Seq 1: 连续分配   │        │                   │
│ [██████          ]│        │ Physical Pages:   │
│     浪费更多 ↑    │        │ P0[██] P1[██]    │
└──────────────────┘        │ P2[██] P3[██]    │
                             │ P4[  ] P5[██]    │
内存碎片严重                  └──────────────────┘
                             无碎片! 按需分配页
```

PagedAttention（vLLM 的核心技术）:
- KV Cache 按固定大小的页 (block) 分配
- 页表记录每个序列的物理页映射
- Attention kernel 通过页表间接寻址
- 极大减少内存浪费，提高推理吞吐

这通常在 vLLM 或 TensorRT-LLM 等推理框架中实现。

## 23.6 注意力优化的未来方向

```
当前技术栈:
├── FlashAttention v1-v3:   精确注意力的极致优化
├── Block-Sparse:           结构化稀疏
├── GQA/MQA:               减少 KV 参数
└── PagedAttention:         推理内存优化

未来方向:
├── 线性注意力 (Linear Attention):
│     复杂度从 O(N²) 降到 O(N)
│     例: Mamba, RWKV, RetNet
│
├── 状态空间模型 (SSM):
│     不需要 attention 矩阵
│     Mamba-2: 混合 SSM + Attention
│
├── 硬件协同设计:
│     Hopper TMA + 注意力 kernel
│     自定义 ASIC for attention
│
└── 算法-硬件联合优化:
      Ring Attention: 多 GPU 分布式注意力
      Sequence Parallelism
```

## 23.7 综合性能对比

In [5]:
# 性能对比: 不同 Attention 实现
def naive_attention(q, k, v, causal=False):
    """标准 attention (第19章)"""
    B, H, N, D = q.shape
    s = q.float() @ k.float().transpose(-2, -1) / math.sqrt(D)
    if causal:
        mask = torch.triu(torch.ones(N, N, device=q.device), diagonal=1).bool()
        s.masked_fill_(mask, float('-inf'))
    p = torch.softmax(s, dim=-1)
    return (p @ v.float()).to(q.dtype)

# 对比不同序列长度
B, H, D = 2, 8, 64
print(f"{'序列长度':<10} {'Naive (ms)':<15} {'GQA FA (ms)':<15} {'加速比':<10}")
print("─" * 50)

for N in [128, 256, 512, 1024]:
    q = torch.randn(B, H, N, D, device='cuda', dtype=torch.float16)
    k = torch.randn(B, H // 4, N, D, device='cuda', dtype=torch.float16)  # GQA: 4x fewer KV heads
    v = torch.randn(B, H // 4, N, D, device='cuda', dtype=torch.float16)
    
    # Naive (expand KV heads for fair comparison)
    k_exp = k.repeat_interleave(4, dim=1)
    v_exp = v.repeat_interleave(4, dim=1)
    ms_naive = triton.testing.do_bench(lambda: naive_attention(q, k_exp, v_exp, causal=True))
    
    # GQA FlashAttention
    ms_gqa = triton.testing.do_bench(lambda: flash_attention_gqa(q, k, v, causal=True))
    
    speedup = ms_naive / ms_gqa
    print(f"{N:<10} {ms_naive:<15.3f} {ms_gqa:<15.3f} {speedup:<10.2f}x")

序列长度       Naive (ms)      GQA FA (ms)     加速比       
──────────────────────────────────────────────────
128        0.101           0.007           13.58     x
256        0.083           0.010           8.52      x
512        0.116           0.014           8.36      x
1024       0.262           0.029           9.10      x


## 教程完结总结

恭喜你完成了 **Triton 从入门到专家** 的全部 26 章教程！

### 你的学习路径

```
Part 1: 基础 (6章)           Part 2: 核心模式 (5章)
━━━━━━━━━━━━━━━━━━          ━━━━━━━━━━━━━━━━━━━━━
✅ 向量加法                  ✅ Softmax (在线算法)
✅ 执行模型                  ✅ 朴素矩阵乘法
✅ 内存操作                  ✅ 分块矩阵乘法
✅ Block 操作                ✅ 算子融合
✅ 控制流与归约              ✅ 自动调优
✅ 数学运算

Part 3: GEMM 优化 (7章)      Part 4: FlashAttention (5章)
━━━━━━━━━━━━━━━━━━━━        ━━━━━━━━━━━━━━━━━━━━━━━━
✅ 共享内存 (Block Ptr)      ✅ 标准 Attention (O(N²))
✅ Swizzle (L2 缓存)        ✅ FlashAttention v1
✅ 软件流水线                ✅ FlashAttention v2
✅ Split-K 并行             ✅ FlashAttention v3
✅ 混合精度                  ✅ FlashAttention v4 + GQA
✅ Tensor Core 深入
✅ 终极优化 GEMM

Part 5: 高级 (3章)
━━━━━━━━━━━━━━━━━━
✅ 调试与性能分析
✅ 编译器 IR
✅ 生产集成
```

### CUDA → Triton 对应关系

M=N=2048, K=1024, RTX PRO 6000 Blackwell (SM 12.0, 188 SMs) 实测:

| CUDA 项目 Kernel | 性能 | TFLOPS | Triton 对应章节 |
|:---|:---:|:---:|:---|
| simt_naive | 6.7966 ms | 1.26 | 第08章 朴素 GEMM |
| simt_regci | 0.3833 ms | 22.41 | 第09章 分块 GEMM |
| simt_smem | 0.2518 ms | 34.11 | 第12章 共享内存 |
| simt_smemT | 0.2349 ms | 36.57 | 第13章 Swizzle |
| simt_pipline | 0.2410 ms | 35.65 | 第14章 流水线 |
| wmma_ci | 0.1556 ms | 55.22 | 第17章 Tensor Core |
| mma_pipline | 0.1844 ms | 46.59 | 第18章 终极 GEMM |
| cuBLAS | 0.1384 ms | 62.05 | 基准 |

### 继续学习
- 阅读 [Triton 官方教程](https://triton-lang.org/main/getting-started/tutorials/)
- 研究 [FlashAttention 源码](https://github.com/Dao-AILab/flash-attention)
- 贡献 Triton 生态: 实现更多自定义 kernel
- 关注最新进展: Triton 3.0, FlashAttention 3/4

## 练习

1. 实现完整的 varlen FlashAttention Triton kernel
2. 实现滑动窗口 + 因果 mask 的 FlashAttention
3. 用 block mask 实现块稀疏 attention
4. 将 GQA FlashAttention 封装为 `torch.autograd.Function`，支持训练